# Zero-Sum Linear Program

In [6]:
import numpy as np
import pyomo.environ as pyo


def build_linear_zero_sum_model(
    A_bar: np.ndarray,
) -> pyo.ConcreteModel:

    model = pyo.ConcreteModel()

    # Sets
    model.I = pyo.RangeSet(0, A_bar.shape[0] - 1)
    model.J = pyo.RangeSet(0, A_bar.shape[1] - 1)

    # Parameters
    model.A_bar = pyo.Param(
        model.I,
        model.J,
        initialize={(i, j): A_bar[i, j] for i in model.I for j in model.J},
        domain=pyo.Reals,
    )

    # Variables
    model.x = pyo.Var(model.I, domain=pyo.NonNegativeReals)
    model.pi1 = pyo.Var(domain=pyo.Reals)

    # Objective
    model.obj = pyo.Objective(
        expr=model.pi1,
        sense=pyo.maximize,
    )

    # Constraints
    model.game_value = pyo.Constraint(
        model.J,
        rule=lambda m, j: sum(model.x[i] * m.A_bar[i, j] for i in model.I) >= m.pi1,
    )
    model.simplex_x = pyo.Constraint(expr=sum(model.x[i] for i in model.I) == 1.0)

    # import duals for constraints
    model.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

    return model


def solve_model(m: pyo.ConcreteModel, solver: str = "cbc"):
    opt = pyo.SolverFactory(solver)
    if opt.available(exception_flag=False):
        res = opt.solve(m, tee=True)
        return res


def build_and_solve(A_bar):
    x_mod = build_linear_zero_sum_model(A_bar)
    solve_model(x_mod)

    # Column strategy y = duals of col constraints
    y = np.array([x_mod.dual[x_mod.game_value[j]] for j in x_mod.J])
    # numerical tidy
    y = np.maximum(y, 0)
    s = y.sum()
    if s > 0:
        y /= s

    # extract
    x = np.array([pyo.value(x_mod.x[i]) for i in x_mod.I])
    y = np.array([pyo.value(y[i]) for i in range(len(y))])

    return {
        "x": x,
        "y": y,
        "game_value": pyo.value(x_mod.obj),
    }


if __name__ == "__main__":
    # Demo
    A1 = np.array([[5, 4], [2, 3]])
    A2 = np.array([[3, 4], [6, 5]])
    B1 = np.array([[5, 2], [4, 3]])
    B2 = np.array([[3, 6], [4, 5]])
    A_list = [A1, A2]
    B_list = [B1, B2]
    p = np.array([0.5, 0.5])

    A_bar = np.array(sum([pk * A for pk, A in zip(p, A_list)]))
    B_bar = np.array(sum([pk * B for pk, B in zip(p, B_list)]))

    print("A_bar =\n", A_bar)
    print("B_bar =\n", B_bar)

    res = build_and_solve(A_bar)
    print("x* =", res["x"], " y* =", res["y"], " game value =", res["game_value"])

A_bar =
 [[4. 4.]
 [4. 4.]]
B_bar =
 [[4. 4.]
 [4. 4.]]
Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - /opt/homebrew/opt/cbc/bin/cbc -printingOptions all -import /var/folders/bq/rft0jdgx1kj30hkw43yb62wm0000gn/T/tmptv36yz1a.pyomo.lp -stat=1 -solve -solu /var/folders/bq/rft0jdgx1kj30hkw43yb62wm0000gn/T/tmptv36yz1a.pyomo.soln (default strategy 1)
Option for printingOptions changed from normal to all
 CoinLpIO::readLp(): Maximization problem reformulated as minimization
Coin0009I Switching back to maximization to get correct duals etc
Presolve 0 (-3) rows, 0 (-3) columns and 0 (-8) elements
Statistics for presolved model


Problem has 0 rows, 0 columns (0 with objective) and 0 elements
Column breakdown:
0 of type 0.0->inf, 0 of type 0.0->up, 0 of type lo->inf, 
0 of type lo->up, 0 of type free, 0 of type fixed, 
0 of type -inf->0.0, 0 of type -inf->up, 0 of type 0.0->1.0 
Row breakdown:
0 of type E 0.0, 0 of type E 1.0, 0 of type E -1.0, 
0 of ty

# Pyomo Complementarity

In [7]:
# Two-player mean-semi-deviation (MSD) risk-aware bimatrix game as a Multilinear Complementarity Problem (MCP)
# Requires: pyomo (>=6), plus PATH ('path' or 'pathmcp') or IPOPT with mpec.simple_nonlinear transform.


import numpy as np
import pyomo.environ as pyo
from pyomo.mpec import Complementarity, complements


def build_model(
    A_list: list[np.ndarray], B_list: list[np.ndarray], p: np.ndarray, gamma: float
) -> pyo.ConcreteModel:
    K = len(A_list)
    n1, n2 = A_list[0].shape
    assert all(Ak.shape == (n1, n2) for Ak in A_list)
    assert all(Bk.shape == (n1, n2) for Bk in B_list)
    p = np.asarray(p, dtype=float)
    p = p / p.sum()

    m = pyo.ConcreteModel()

    # Sets
    m.I = pyo.RangeSet(0, n1 - 1)
    m.J = pyo.RangeSet(0, n2 - 1)
    m.K = pyo.RangeSet(0, K - 1)

    # Params (store as dicts for Pyomo)
    # Converts payoff matrices to 3D arrays
    def to_param_3d(lst):
        data = {}
        for k, M in enumerate(lst):
            for i in range(n1):
                for j in range(n2):
                    data[(k, i, j)] = float(M[i, j])
        return data

    m.p = pyo.Param(
        m.K, initialize={k: float(p[k]) for k in range(K)}, within=pyo.NonNegativeReals
    )
    m.gamma = pyo.Param(initialize=float(gamma), within=pyo.NonNegativeReals)
    m.A = pyo.Param(m.K, m.I, m.J, initialize=to_param_3d(A_list))
    m.B = pyo.Param(m.K, m.I, m.J, initialize=to_param_3d(B_list))

    # Abar, Bbar
    def Abar_rule(m, i, j):
        return sum(m.p[k] * m.A[k, i, j] for k in m.K)

    def Bbar_rule(m, i, j):
        return sum(m.p[k] * m.B[k, i, j] for k in m.K)

    m.Abar = pyo.Param(
        m.I,
        m.J,
        initialize={(i, j): Abar_rule(m, i, j) for i in range(n1) for j in range(n2)},
    )
    m.Bbar = pyo.Param(
        m.I,
        m.J,
        initialize={(i, j): Bbar_rule(m, i, j) for i in range(n1) for j in range(n2)},
    )

    # Variables
    m.x = pyo.Var(m.I, domain=pyo.NonNegativeReals)  # strategies
    m.y = pyo.Var(m.J, domain=pyo.NonNegativeReals)
    m.alpha1 = pyo.Var(domain=pyo.Reals)  # values
    m.alpha2 = pyo.Var(domain=pyo.Reals)
    m.lam1 = pyo.Var(
        m.K, bounds=lambda m, k: (0.0, pyo.value(m.gamma) * pyo.value(m.p[k]))
    )
    m.lam2 = pyo.Var(
        m.K, bounds=lambda m, k: (0.0, pyo.value(m.gamma) * pyo.value(m.p[k]))
    )
    m.z1 = pyo.Var(m.K, domain=pyo.NonNegativeReals)  # z >= 0
    m.z2 = pyo.Var(m.K, domain=pyo.NonNegativeReals)

    # Non-zero solution constraint to avoid trivial all-zero solution if using the non-simplex formulation
    # m.nonzero_x = pyo.Constraint(expr=sum(m.x[i] for i in m.I) >= 1e-6)
    # m.nonzero_y = pyo.Constraint(expr=sum(m.y[j] for j in m.J) >= 1e-6)
    # m.nonzero_z1 = pyo.Constraint(expr=sum(m.z1[k] for k in m.K) >= 1e-6)
    # m.nonzero_z2 = pyo.Constraint(expr=sum(m.z2[k] for k in m.K) >= 1e-6)

    # Simplex equalities
    m.simplex_x = pyo.Constraint(expr=sum(m.x[i] for i in m.I) == 1.0)
    m.simplex_y = pyo.Constraint(expr=sum(m.y[j] for j in m.J) == 1.0)

    # Helper expressions
    # Player 1 row values: v1[i] = Abar[i,:] y + sum_k lam1[k]*(A^k - Abar)[i,:] y
    def v1_rule(m, i):
        base = sum(m.Abar[i, j] * m.y[j] for j in m.J)
        dev = sum(
            m.lam1[k] * sum((m.A[k, i, j] - m.Abar[i, j]) * m.y[j] for j in m.J)
            for k in m.K
        )
        return base + dev

    m.v1 = pyo.Expression(m.I, rule=v1_rule)

    # Player 2 column values: v2[j] = x^T Bbar[:,j] + sum_k lam2[k]* x^T (B^k - Bbar)[:,j]
    def v2_rule(m, j):
        base = sum(m.x[i] * m.Bbar[i, j] for i in m.I)
        dev = sum(
            m.lam2[k] * sum(m.x[i] * (m.B[k, i, j] - m.Bbar[i, j]) for i in m.I)
            for k in m.K
        )
        return base + dev

    m.v2 = pyo.Expression(m.J, rule=v2_rule)

    # Scenario residuals:
    # r1[k] = x^T (A^k - Abar) y
    def r1_rule(m, k):
        return sum(
            m.x[i] * sum((m.A[k, i, j] - m.Abar[i, j]) * m.y[j] for j in m.J)
            for i in m.I
        )

    m.r1 = pyo.Expression(m.K, rule=r1_rule)

    # r2[k] = x^T (B^k - Bbar) y
    def r2_rule(m, k):
        return sum(
            m.x[i] * sum((m.B[k, i, j] - m.Bbar[i, j]) * m.y[j] for j in m.J)
            for i in m.I
        )

    m.r2 = pyo.Expression(m.K, rule=r2_rule)

    # Complementarity conditions
    # 0 <= x_i ⟂ alpha1 - v1_i >= 0
    m.comp_x = Complementarity(
        m.I, rule=lambda m, i: complements(m.x[i] >= 0, m.alpha1 - m.v1[i] >= 0)
    )
    # 0 <= y_j ⟂ alpha2 - v2_j >= 0
    m.comp_y = Complementarity(
        m.J, rule=lambda m, j: complements(m.y[j] >= 0, m.alpha2 - m.v2[j] >= 0)
    )
    # 0 <= lam1_k ⟂ r1_k + z1_k >= 0 ; 0 <= gamma p_k - lam1_k ⟂ z1_k >= 0
    m.comp_r1 = Complementarity(
        m.K, rule=lambda m, k: complements(m.lam1[k] >= 0, m.r1[k] + m.z1[k] >= 0)
    )
    m.comp_z1 = Complementarity(
        m.K,
        rule=lambda m, k: complements(m.gamma * m.p[k] - m.lam1[k] >= 0, m.z1[k] >= 0),
    )
    # Player 2 analogues
    m.comp_r2 = Complementarity(
        m.K, rule=lambda m, k: complements(m.lam2[k] >= 0, m.r2[k] + m.z2[k] >= 0)
    )
    m.comp_z2 = Complementarity(
        m.K,
        rule=lambda m, k: complements(m.gamma * m.p[k] - m.lam2[k] >= 0, m.z2[k] >= 0),
    )

    return m


def solve_model(m: pyo.ConcreteModel, solver: str = "path"):
    opt = pyo.SolverFactory(solver)
    if opt.available(exception_flag=False):
        res = opt.solve(m, tee=True)
        return res
    # Fallback NLP transform
    pyo.TransformationFactory("mpec.simple_nonlinear").apply_to(m)
    opt = pyo.SolverFactory("ipopt")
    assert opt.available(exception_flag=False), "Neither PATH nor IPOPT available."
    res = opt.solve(m, tee=True)
    return res


def build_and_solve(A_list, B_list, p, gamma, solver="pathampl"):
    m = build_model(A_list, B_list, p, gamma)
    solve_model(m, solver=solver)
    # extract
    x = np.array([pyo.value(m.x[i]) for i in m.I])
    y = np.array([pyo.value(m.y[j]) for j in m.J])
    alpha1 = float(pyo.value(m.alpha1))
    alpha2 = float(pyo.value(m.alpha2))
    lam1 = np.array([pyo.value(m.lam1[k]) for k in m.K])
    lam2 = np.array([pyo.value(m.lam2[k]) for k in m.K])
    z1 = np.array([pyo.value(m.z1[k]) for k in m.K])
    z2 = np.array([pyo.value(m.z2[k]) for k in m.K])
    return {
        "x": x,
        "y": y,
        "alpha1": alpha1,
        "alpha2": alpha2,
        "lam1": lam1,
        "z1": z1,
        "lam2": lam2,
        "z2": z2,
    }


if __name__ == "__main__":
    # Demo
    A1 = np.array([[5, 4], [2, 3]])
    A2 = np.array([[3, 4], [6, 5]])
    B1 = np.array([[5, 2], [4, 3]])
    B2 = np.array([[3, 6], [4, 5]])
    A_list = [A1, A2]
    B_list = [B1, B2]
    p = np.array([0.5, 0.5])
    gamma = 1
    res = build_and_solve(A_list, B_list, p, gamma)
    print("x* =", res["x"], " y* =", res["y"])
    print("alpha1 =", res["alpha1"], " alpha2 =", res["alpha2"])
    print("lam1 =", res["lam1"], " z1 =", res["z1"])
    print("lam2 =", res["lam2"], " z2 =", res["z2"])

Path 5.0.05: Path 5.0.05 (Wed Jun 23 14:34:38 2021)
Written by Todd Munson, Steven Dirkse, Youngdae Kim, and Michael Ferris


INITIAL POINT STATISTICS
Maximum of X. . . . . . . . . .  0.0000e+00 var: (_svar[1])
Maximum of F. . . . . . . . . .  1.0000e+00 eqn: (_scon[19])
Maximum of Grad F . . . . . . .  4.0000e+00 eqn: (_scon[11])
                                            var: (_svar[1])

INITIAL JACOBIAN NORM STATISTICS
Maximum Row Norm. . . . . . . .  1.0000e+01 eqn: (_scon[9])
Minimum Row Norm. . . . . . . .  1.0000e+00 eqn: (_scon[1])
Maximum Column Norm . . . . . .  9.0000e+00 var: (_svar[1])
Minimum Column Norm . . . . . .  1.0000e+00 var: (_svar[5])

Crash Log
major  func  diff  size  residual    step       prox   (label)
    0     0             1.7321e+00             0.0e+00 (_scon[19])
pn_search terminated: no progress.

Major Iteration Log
major minor  func  grad  residual    step  type prox    inorm  (label)
    0     0    13     1 1.7321e+00           I 1.7e-02 1.0e+00 (_

# Fischer-Burmeister (FB) NCP Solver

In [ ]:
# Two-player MSD risk-aware game solved as an MCP via a Fischer–Burmeister (FB) NCP solver.
# This implements the multilinear complementarity program directly (not an LCP).
# It uses SciPy's least_squares to drive the FB residual to zero with simple bounds.

from typing import Any

import numpy as np

try:
    from scipy.optimize import least_squares

    SCIPY_OK = True
except Exception:
    SCIPY_OK = False


def fb(a, b):
    """Fischer–Burmeister function φ(a,b) = sqrt(a^2+b^2) - (a+b). Zero iff a>=0, b>=0, a*b=0."""
    return np.sqrt(a * a + b * b) - (a + b)


def pack_vars(x, y, lam1, z1, lam2, z2, alpha1, alpha2):
    return np.concatenate([x, y, lam1, z1, lam2, z2, np.array([alpha1, alpha2])])


def unpack_vars(v, n1, n2, K):
    x = v[0:n1]
    y = v[n1 : n1 + n2]
    lam1 = v[n1 + n2 : n1 + n2 + K]
    z1 = v[n1 + n2 + K : n1 + n2 + 2 * K]
    lam2 = v[n1 + n2 + 2 * K : n1 + n2 + 3 * K]
    z2 = v[n1 + n2 + 3 * K : n1 + n2 + 4 * K]
    alpha1 = v[-2]
    alpha2 = v[-1]
    return x, y, lam1, z1, lam2, z2, alpha1, alpha2


def mcp_residual(v, A_list, B_list, p, gamma):
    """Build FB residual vector F(v)=0 for the MSD bimatrix MCP."""
    n1, n2 = A_list[0].shape
    K = len(A_list)
    x, y, lam1, z1, lam2, z2, alpha1, alpha2 = unpack_vars(v, n1, n2, K)

    # Abar, Bbar
    Abar = sum(pk * Ak for pk, Ak in zip(p, A_list))
    Bbar = sum(pk * Bk for pk, Bk in zip(p, B_list))

    # Player 1: row values v1_l(y, lam1)
    # v1_l = (Abar_l + Σ λ_k (A^k_l - Abar_l)) y == Abar_l y + Σ λ_k (A^k_l - Abar_l) y
    # We'll compute matrix V1 := Abar + Σ λ_k (A^k - Abar)  so that v1 = V1 y
    V1 = (
        Abar.copy()
    )  # start from Abar then add weighted deviations (since stationarity uses Abar)
    for k in range(K):
        V1 += lam1[k] * (A_list[k] - Abar)
    v1_rows = V1 @ y  # shape (n1,)

    # Player 2: column values v2_c(x, lam2) with V2^T x
    V2 = Bbar.copy()
    for k in range(K):
        V2 += lam2[k] * (B_list[k] - Bbar)
    v2_cols = V2.T @ x  # shape (n2,)

    # Scenario residuals
    r1 = np.array([x @ ((A_list[k] - Abar) @ y) for k in range(K)])
    r2 = np.array([x @ ((B_list[k] - Bbar) @ y) for k in range(K)])

    # Build FB residuals
    res = []

    # (i) Strategy–value complementarity
    # 0 <= x ⊥ (alpha1 - v1_rows) >= 0
    res.extend(fb(x, alpha1 - v1_rows))
    # 0 <= y ⊥ (alpha2 - v2_cols) >= 0
    res.extend(fb(y, alpha2 - v2_cols))

    # (ii) MSD residual pairs
    # For player 1:
    # 0 <= lam1 ⊥ (r1 - z1) >= 0
    res.extend(fb(lam1, r1 - z1))
    # 0 <= gamma*p - lam1 ⊥ (-z1) >= 0
    res.extend(fb(gamma * p - lam1, -z1))

    # For player 2:
    res.extend(fb(lam2, r2 - z2))
    res.extend(fb(gamma * p - lam2, -z2))

    # (iii) Simplex equalities (sum x = 1, sum y = 1)
    res.append(x.sum() - 1.0)
    res.append(y.sum() - 1.0)

    return np.array(res, dtype=float)


def solve_mcp_msd_bimatrix(
    A_list: list[np.ndarray],
    B_list: list[np.ndarray],
    p: np.ndarray,
    gamma: float,
    x0: np.ndarray = None,
    y0: np.ndarray = None,
    max_nfev: int = 20000,
    verbose: bool = True,
) -> dict[str, Any]:
    """Solve the MSD bimatrix MCP via FB least-squares."""
    assert SCIPY_OK, "SciPy is required for this solver."
    K = len(A_list)
    n1, n2 = A_list[0].shape
    p = np.asarray(p, dtype=float)
    p = p / p.sum()

    # Initial strategies
    if x0 is None:
        x0 = np.ones(n1) / n1
    if y0 is None:
        y0 = np.ones(n2) / n2

    # Initialize lambdas, z, alphas
    lam1 = np.zeros(K)
    lam2 = np.zeros(K)
    z1 = np.zeros(K)
    z2 = np.zeros(K)

    # Reasonable alpha in [0,1+gamma]: expected value + gamma * expected downside (<= gamma)
    Abar = sum(pk * Ak for pk, Ak in zip(p, A_list))
    Bbar = sum(pk * Bk for pk, Bk in zip(p, B_list))
    alpha1 = float(x0 @ (Abar @ y0))
    alpha2 = float(x0 @ (Bbar @ y0))

    v0 = pack_vars(x0, y0, lam1, z1, lam2, z2, alpha1, alpha2)

    # Bounds: x,y in [0,1]; lam in [0, gamma p_k]; z in (-inf, 0]; alpha in [-10, 10]
    lb = np.concatenate(
        [
            np.zeros(n1),
            np.zeros(n2),
            np.zeros(K),
            -np.full(K, np.inf),
            np.zeros(K),
            -np.full(K, np.inf),
            np.array([-np.inf, -np.inf]),
        ]
    )
    ub = np.concatenate(
        [
            np.ones(n1),
            np.ones(n2),
            gamma * p,
            np.zeros(K),
            gamma * p,
            np.zeros(K),
            np.array([np.inf, np.inf]),
        ]
    )

    fun = lambda v: mcp_residual(v, A_list, B_list, p, gamma)
    res = least_squares(
        fun,
        v0,
        jac="3-point",
        bounds=(lb, ub),
        xtol=1e-12,
        ftol=1e-12,
        gtol=1e-12,
        max_nfev=max_nfev,
        verbose=2 if verbose else 0,
    )

    v_star = res.x
    x, y, lam1, z1, lam2, z2, alpha1, alpha2 = unpack_vars(v_star, n1, n2, K)

    out = {
        "success": res.success,
        "status": res.status,
        "message": res.message,
        "residual_norm": np.linalg.norm(res.fun),
        "x": x,
        "y": y,
        "alpha1": float(alpha1),
        "alpha2": float(alpha2),
        "lam1": lam1,
        "z1": z1,
        "lam2": lam2,
        "z2": z2,
        "nfev": res.nfev,
    }
    return out


# ----------- Demo on a small instance -----------


def demo_mcp():
    A1 = np.array([[0.8, 0.1], [0.2, 0.6]])
    A2 = np.array([[0.3, -0.9], [-0.7, 0.4]]) / 0.25
    B1 = -np.array([[0.8, 0.1], [0.2, 0.6]])
    B2 = -np.array([[0.3, -0.9], [-0.7, 0.4]]) / 0.25
    A_list = [A1, A2]
    B_list = [B1, B2]
    p = np.array([0.75, 0.25])
    gamma = 1e-12

    sol = solve_mcp_msd_bimatrix(A_list, B_list, p, gamma, verbose=True)
    print("\n=== MCP Result ===")
    for k, v in sol.items():
        if isinstance(v, np.ndarray):
            print(f"{k} = {np.round(v, 6)}")
        else:
            print(f"{k} = {v}")


demo_mcp()

# Alternating LP using SciPy and projected subgradient

In [ ]:
# Two-player MSD risk-aware game solver via alternating LP best-responses
# Falls back to projected subgradient BR if SciPy isn't available.


import numpy as np

# ---------- Utilities ----------


def project_to_simplex(v: np.ndarray) -> np.ndarray:
    """Euclidean projection of v onto the probability simplex {x>=0, sum x = 1}."""
    if np.all(v >= 0) and abs(v.sum() - 1.0) < 1e-12:
        return v.copy()
    n = v.size
    u = np.sort(v)[::-1]
    cssv = np.cumsum(u)
    rho = np.nonzero(u * np.arange(1, n + 1) > (cssv - 1))[0][-1]
    theta = (cssv[rho] - 1) / (rho + 1.0)
    w = np.maximum(v - theta, 0.0)
    return w


def compute_values(
    A_list: list[np.ndarray],
    B_list: list[np.ndarray],
    p: np.ndarray,
    gamma: float,
    x: np.ndarray,
    y: np.ndarray,
) -> tuple[float, float]:
    """Risk-adjusted values (alpha^1, alpha^2) at (x,y)."""
    Abar = sum(pk * Ak for pk, Ak in zip(p, A_list))
    Bbar = sum(pk * Bk for pk, Bk in zip(p, B_list))
    val1 = float(x @ (Abar @ y))
    val2 = float(x @ (Bbar @ y))
    # MSD penalties (negative semideviation part)
    for pk, Ak, Bk in zip(p, A_list, B_list):
        d1 = float(x @ ((Ak - Abar) @ y))
        d2 = float(x @ ((Bk - Bbar) @ y))
        val1 += gamma * pk * min(0.0, d1)
        val2 += gamma * pk * min(0.0, d2)
    return val1, val2


# ---------- LP Best-Responses (SciPy) ----------

try:
    from scipy.optimize import linprog

    SCIPY_OK = True
except Exception:
    SCIPY_OK = False


def br1_lp(A_list, p, gamma, y) -> np.ndarray | None:
    """Player 1 best response via LP, given y. Returns x or None if LP not available/fails."""
    if not SCIPY_OK:
        return None
    n1 = A_list[0].shape[0]
    K = len(A_list)
    Abar_y = sum(pk * (Ak @ y) for pk, Ak in zip(p, A_list))  # n1-vector
    dky_list = [(Ak @ y - Abar_y) for Ak in A_list]  # K of n1-vectors

    # Variables order: [x (n1), z (K)]
    c = np.concatenate([-Abar_y, -gamma * p])  # maximize => minimize negative
    A_ub = []
    b_ub = []
    # z_k - d_k·x <= 0
    for dk in dky_list:
        row = np.concatenate([-dk, np.eye(1, K, A_ub.__len__()).ravel()])
        A_ub.append(row)
        b_ub.append(0.0)
    # z_k <= 0
    for k in range(K):
        row = np.concatenate([np.zeros(n1), np.eye(1, K, k).ravel()])
        A_ub.append(row)
        b_ub.append(0.0)
    A_ub = np.vstack(A_ub)

    b_ub = np.array(b_ub)
    # Equality: sum x = 1
    A_eq = np.concatenate([np.ones(n1), np.zeros(K)])[None, :]
    b_eq = np.array([1.0])
    # Bounds: x_j >= 0 ; z_k free below, <=0
    bounds = [(0.0, None)] * n1 + [(None, 0.0)] * K
    res = linprog(
        c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs"
    )
    if not res.success:
        return None

    return res.x


def br2_lp(B_list, p, gamma, x) -> np.ndarray | None:
    """Player 2 best response via LP, given x. Returns y or None if LP not available/fails."""
    if not SCIPY_OK:
        return None
    n2 = B_list[0].shape[1]
    K = len(B_list)
    xTB = [x @ Bk for Bk in B_list]  # K of n2-rows
    xTBbar = sum(pk * (x @ Bk) for pk, Bk in zip(p, B_list))  # 1×n2 row
    # Variables order: [y (n2), z (K)]
    c = np.concatenate([-xTBbar, -gamma * p])  # maximize => minimize negative
    A_ub = []
    b_ub = []
    # z_k - (x^T(B_k - Bbar)) y <= 0  => row: -r_k for y, 1 for z_k
    for k in range(K):
        rk = xTB[k] - xTBbar  # 1×n2 row
        row = np.concatenate([-rk, np.eye(1, K, k).ravel()])
        A_ub.append(row)
        b_ub.append(0.0)
    # z_k <= 0
    for k in range(K):
        row = np.concatenate([np.zeros(n2), np.eye(1, K, k).ravel()])
        A_ub.append(row)
        b_ub.append(0.0)
    A_ub = np.vstack(A_ub)
    b_ub = np.array(b_ub)
    # Equality: sum y = 1
    A_eq = np.concatenate([np.ones(n2), np.zeros(K)])[None, :]
    b_eq = np.array([1.0])
    # Bounds
    bounds = [(0.0, None)] * n2 + [(None, 0.0)] * K
    res = linprog(
        c, A_ub=A_ub, b_ub=b_ub, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method="highs"
    )
    if not res.success:
        return None

    return res.x


# ---------- Projected Subgradient Best-Responses (fallback) ----------


def br1_subgrad(A_list, p, gamma, y, iters=400, eta0=0.5) -> np.ndarray:
    n1 = A_list[0].shape[0]
    x = np.ones(n1) / n1
    Abar_y = sum(pk * (Ak @ y) for pk, Ak in zip(p, A_list))
    dky = [(Ak @ y - Abar_y) for Ak in A_list]
    for t in range(1, iters + 1):
        g = Abar_y.copy()
        for pk, dk in zip(p, dky):
            if float(dk @ x) < 0:  # subgradient of min(0, dk·x) is dk; else 0
                g += gamma * pk * dk
        eta = eta0 / np.sqrt(t)
        x = project_to_simplex(x + eta * g)
    return x


def br2_subgrad(B_list, p, gamma, x, iters=400, eta0=0.5) -> np.ndarray:
    n2 = B_list[0].shape[1]
    y = np.ones(n2) / n2
    xTBbar = sum(pk * (x @ Bk) for pk, Bk in zip(p, B_list))
    xTB_minus_bar = [(x @ Bk - xTBbar) for Bk in B_list]
    for t in range(1, iters + 1):
        g = xTBbar.copy()
        for pk, rk in zip(p, xTB_minus_bar):
            if float(rk @ y) < 0:
                g += gamma * pk * rk
        eta = eta0 / np.sqrt(t)
        y = project_to_simplex(y + eta * g)
    return y


# ---------- Outer solver ----------


def solve_msd_bimatrix(
    A_list: list[np.ndarray],
    B_list: list[np.ndarray],
    p: np.ndarray,
    gamma: float,
    max_iter: int = 200,
    tol: float = 1e-8,
    verbose: bool = True,
    use_lp: bool = True,
):
    """Alternating best-responses for MSD-risk two-player games.

    Inputs:
      - A_list, B_list: lists of K scenario payoff matrices (n1×n2)
      - p: scenario probabilities (K,), sum=1
      - gamma: MSD risk parameter (>=0; recommend <=1)
    Returns:
      dict with x,y, alpha1, alpha2, history
    """
    K = len(A_list)
    n1, n2 = A_list[0].shape
    assert all(Ak.shape == (n1, n2) for Ak in A_list)
    assert all(Bk.shape == (n1, n2) for Bk in B_list)
    p = np.asarray(p, dtype=float)
    p = p / p.sum()

    # init uniform
    x = np.ones(n1) / n1
    y = np.ones(n2) / n2

    hist = []
    for it in range(1, max_iter + 1):
        x_prev, y_prev = x.copy(), y.copy()

        # Player 1 BR
        if use_lp:
            x_lp = br1_lp(A_list, p, gamma, y)
        else:
            x_lp = None

        if x_lp is not None:
            x, z1 = x_lp[:n1], x_lp[n1:]
        else:
            x = br1_subgrad(A_list, p, gamma, y)
            z1 = np.nan

        # Player 2 BR
        if use_lp:
            y_lp = br2_lp(B_list, p, gamma, x)
        else:
            y_lp = None

        if y_lp is not None:
            y, z2 = y_lp[:n2], y_lp[n2:]
        else:
            y = br2_subgrad(B_list, p, gamma, x)
            z2 = np.nan

        alpha1, alpha2 = compute_values(A_list, B_list, p, gamma, x, y)
        hist.append((x.copy(), y.copy(), alpha1, alpha2))

        dx = np.linalg.norm(x - x_prev, 1)
        dy = np.linalg.norm(y - y_prev, 1)
        if verbose:
            print(
                f"Iter {it:03d}: |dx|_1={dx:.3e}, |dy|_1={dy:.3e}, alpha1={alpha1:.6f}, alpha2={alpha2:.6f}"
            )
        if max(dx, dy) < tol:
            break

    return {
        "x": x,
        "y": y,
        "alpha1": alpha1,
        "alpha2": alpha2,
        "z1": z1,
        "z2": z2,
        "history": hist,
        "used_lp": SCIPY_OK and use_lp,
    }


# ---------- Demo ----------


def demo():
    A1 = np.array([[0.8, 0.1], [0.2, 0.6]])
    A2 = np.array([[0.3, -0.9], [-0.7, 0.4]]) / 0.25
    B1 = -np.array([[0.8, 0.1], [0.2, 0.6]])
    B2 = -np.array([[0.3, -0.9], [-0.7, 0.4]]) / 0.25
    A_list = [A1, A2]
    B_list = [B1, B2]
    p = np.array([0.75, 0.25])
    gamma = 0.0

    res = solve_msd_bimatrix(
        A_list, B_list, p, gamma, max_iter=201, tol=1e-10, verbose=True, use_lp=True
    )
    print("\n=== Result ===")
    print(f"Used LP: {res['used_lp']}")
    print("x* =", np.round(res["x"], 6))
    print("y* =", np.round(res["y"], 6))
    print("z1* =", np.round(res["z1"], 6))
    print("z2* =", np.round(res["z2"], 6))
    print("alpha1 =", round(res["alpha1"], 6))
    print("alpha2 =", round(res["alpha2"], 6))


demo()

# N-Player Extension

In [ ]:
# Extended data structures for N-player MSD games

from itertools import product

import numpy as np


class NPlayerMSDGame:
    """N-player Mean-Semi-Deviation risk-aware game solver."""

    def __init__(
        self, payoff_tensors: list[list[np.ndarray]], p: np.ndarray, gamma: float
    ):
        """
        Args:
            payoff_tensors: List of K scenarios, each containing N payoff tensors
                          payoff_tensors[k][i] is player i's payoff tensor for scenario k
                          Shape: (n_1, n_2, ..., n_N) for each tensor
            p: Scenario probabilities (K,)
            gamma: MSD risk parameter
        """
        self.K = len(payoff_tensors)  # number of scenarios
        self.N = len(payoff_tensors[0])  # number of players
        self.payoff_tensors = payoff_tensors
        self.p = np.asarray(p) / np.sum(p)
        self.gamma = gamma

        # Strategy space sizes for each player
        self.strategy_sizes = list(payoff_tensors[0][0].shape)
        self.total_strategies = sum(self.strategy_sizes)

        # Compute expected payoff tensors
        self.expected_payoffs = []
        for i in range(self.N):
            expected = sum(self.p[k] * payoff_tensors[k][i] for k in range(self.K))
            self.expected_payoffs.append(expected)

    def compute_player_payoff(
        self, player: int, strategies: list[np.ndarray], scenario: int = None
    ) -> float:
        """Compute payoff for a player given all players' strategies."""
        if scenario is None:
            # Use expected payoffs
            payoff_tensor = self.expected_payoffs[player]
        else:
            payoff_tensor = self.payoff_tensors[scenario][player]

        # Multilinear form: tensor contracted with all strategy vectors
        result = payoff_tensor
        for i, strategy in enumerate(strategies):
            # Contract along axis i with strategy vector
            result = np.tensordot(result, strategy, axes=([0], [0]))

        return float(result)

    def compute_msd_payoff(self, player: int, strategies: list[np.ndarray]) -> float:
        """Compute MSD-adjusted payoff for a player."""
        expected_payoff = self.compute_player_payoff(player, strategies)

        # Add MSD penalty terms
        msd_penalty = 0.0
        for k in range(self.K):
            scenario_payoff = self.compute_player_payoff(player, strategies, scenario=k)
            deviation = scenario_payoff - expected_payoff
            msd_penalty += self.gamma * self.p[k] * min(0.0, deviation)

        return expected_payoff + msd_penalty


# Example usage for 3-player game
def create_3player_example():
    """Create a simple 3-player, 2-scenario game."""
    # Each player has 2 strategies, so payoff tensors are 2×2×2
    np.random.seed(42)

    payoff_tensors = []
    for k in range(2):  # 2 scenarios
        scenario_payoffs = []
        for i in range(3):  # 3 players
            # Random payoff tensor of shape (2, 2, 2)
            tensor = np.random.rand(2, 2, 2)
            scenario_payoffs.append(tensor)
        payoff_tensors.append(scenario_payoffs)

    p = np.array([0.6, 0.4])  # scenario probabilities
    gamma = 0.3

    game = NPlayerMSDGame(payoff_tensors, p, gamma)

    # Test with uniform strategies
    strategies = [np.array([0.5, 0.5]) for _ in range(3)]

    print("3-Player MSD Game Example:")
    print(f"Strategy space sizes: {game.strategy_sizes}")
    print(f"Total strategies: {game.total_strategies}")

    for i in range(3):
        payoff = game.compute_msd_payoff(i, strategies)
        print(f"Player {i + 1} MSD payoff: {payoff:.4f}")


create_3player_example()

In [ ]:
# N-Player MCP Formulation


def build_nplayer_mcp_model(game: NPlayerMSDGame) -> "pyo.ConcreteModel":
    """Build Pyomo MCP model for N-player MSD game."""
    import pyomo.environ as pyo

    m = pyo.ConcreteModel()

    # Sets
    m.N = pyo.RangeSet(0, game.N - 1)  # Players
    m.K = pyo.RangeSet(0, game.K - 1)  # Scenarios

    # Strategy sets for each player (variable sized)
    m.strategy_sets = {}
    for i in range(game.N):
        m.strategy_sets[i] = pyo.RangeSet(0, game.strategy_sizes[i] - 1)

    # Parameters
    m.p = pyo.Param(m.K, initialize={k: game.p[k] for k in range(game.K)})
    m.gamma = pyo.Param(initialize=game.gamma)

    # Store payoff tensors as parameters (flattened for Pyomo)
    m.payoff_data = {}
    for k in range(game.K):
        for i in range(game.N):
            tensor = game.payoff_tensors[k][i]
            # Flatten tensor and store with multi-index
            for idx in product(*[range(s) for s in tensor.shape]):
                m.payoff_data[(k, i) + idx] = float(tensor[idx])

    # Expected payoff tensors
    m.expected_payoff_data = {}
    for i in range(game.N):
        tensor = game.expected_payoffs[i]
        for idx in product(*[range(s) for s in tensor.shape]):
            m.expected_payoff_data[(i,) + idx] = float(tensor[idx])

    # Variables
    # Strategy variables for each player
    m.strategies = {}
    for i in range(game.N):
        m.strategies[i] = pyo.Var(m.strategy_sets[i], domain=pyo.NonNegativeReals)

    # Value variables for each player
    m.values = pyo.Var(m.N, domain=pyo.Reals)

    # MSD dual variables for each player and scenario
    m.lambdas = {}
    m.z_vars = {}
    for i in range(game.N):
        m.lambdas[i] = pyo.Var(m.K, bounds=lambda m, k: (0.0, game.gamma * game.p[k]))
        m.z_vars[i] = pyo.Var(m.K, bounds=(None, 0.0))

    # Simplex constraints for each player
    for i in range(game.N):
        m.add_component(
            f"simplex_{i}",
            pyo.Constraint(
                expr=sum(m.strategies[i][j] for j in m.strategy_sets[i]) == 1.0
            ),
        )

    # Helper function to compute expected payoff for player i at strategy profile
    def compute_expected_value(m, player):
        """Compute expected payoff for player i given current strategies."""
        total = 0.0

        # Iterate over all strategy combinations
        ranges = [m.strategy_sets[j] for j in range(game.N)]
        for strategy_combo in product(*ranges):
            # Get payoff coefficient
            coeff = m.expected_payoff_data[(player,) + strategy_combo]

            # Multiply by product of strategy probabilities
            prob_product = 1.0
            for j, s_idx in enumerate(strategy_combo):
                prob_product *= m.strategies[j][s_idx]

            total += coeff * prob_product

        return total

    # Create expressions for expected payoffs
    m.expected_payoffs_expr = {}
    for i in range(game.N):
        m.expected_payoffs_expr[i] = pyo.Expression(expr=compute_expected_value(m, i))

    # Scenario residuals for each player
    def compute_scenario_residual(m, player, scenario):
        """Compute (scenario_payoff - expected_payoff) for player at scenario."""
        scenario_total = 0.0

        ranges = [m.strategy_sets[j] for j in range(game.N)]
        for strategy_combo in product(*ranges):
            coeff = m.payoff_data[(scenario, player) + strategy_combo]
            prob_product = 1.0
            for j, s_idx in enumerate(strategy_combo):
                prob_product *= m.strategies[j][s_idx]
            scenario_total += coeff * prob_product

        return scenario_total - m.expected_payoffs_expr[player]

    m.residuals = {}
    for i in range(game.N):
        m.residuals[i] = {}
        for k in range(game.K):
            m.residuals[i][k] = pyo.Expression(expr=compute_scenario_residual(m, i, k))

    # Complementarity conditions
    # For each player i and strategy j: 0 <= x_i^j ⟂ (value_i - marginal_payoff_i^j) >= 0
    # This is complex for N-player games due to multilinear payoffs

    # Simplified version: assume we can compute marginal payoffs
    # In practice, this requires careful handling of the multilinear structure

    print(f"Built N-player MCP model with {game.N} players, {game.K} scenarios")
    print(f"Strategy space sizes: {game.strategy_sizes}")
    print(f"Total variables: {sum(game.strategy_sizes) + game.N + 2 * game.N * game.K}")

    return m


# Note: This is a framework - the full implementation requires:
# 1. Proper marginal payoff computation for multilinear forms
# 2. Complementarity conditions for N-player equilibrium
# 3. Efficient handling of the exponentially growing strategy combinations

## Key Challenges in N-Player Extension

### 1. **Computational Complexity**
- **2-player**: O(n₁ × n₂ × K) variables and constraints
- **N-player**: O(∏ᵢnᵢ × K) terms in payoff calculations (exponential growth!)
- **Memory**: Payoff tensors grow exponentially with N

### 2. **Equilibrium Concepts**
- **Nash Equilibrium**: Each player's strategy is optimal given others' strategies
- **Complementarity**: Need N sets of complementarity conditions
- **Uniqueness**: Multiple equilibria become more common

### 3. **Alternative Approaches**

#### A. **Iterative Best Response** (Most Practical)
```python
def n_player_best_response_iteration(game, max_iter=1000):
    strategies = [np.ones(n)/n for n in game.strategy_sizes]  # uniform init
    
    for iteration in range(max_iter):
        for player in range(game.N):
            # Fix other players' strategies, optimize player's strategy
            strategies[player] = solve_single_player_msd_lp(
                game, player, strategies[:player] + strategies[player+1:]
            )
        
        # Check convergence
        if converged(strategies):
            break
    
    return strategies
```

#### B. **Fictitious Play**
- Each player optimizes against historical frequency of opponents' play
- Converges under certain conditions
- Good for large N

#### C. **Evolutionary Approaches**
- Population-based methods
- Replicator dynamics
- Suitable for symmetric games

### 4. **Recommended Implementation Strategy**

1. **Start Small**: Implement for N=3 first
2. **Use Sparse Representations**: Most payoff tensors are sparse
3. **Best Response Iteration**: Most robust for practical problems
4. **Parallel Computing**: Players can compute best responses in parallel